# GameTheory-04h — Jeux de routage / congestion & Price of Anarchy

**Navigation** : [Version C#](GameTheory-04h-Routing-PriceOfAnarchy-CSharp.ipynb) | [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb) | [<< 4g-Supermodularité](GameTheory-04g-Supermodularite-Python.ipynb) | [Index](README.md) | [jumeau Rust](GameTheory-04h-Routing-PriceOfAnarchy-Rust.ipynb)

**Companion Lean** : [GameTheory-04h-Routing-PriceOfAnarchy-Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)

**Objectifs** : calculer un équilibre de Wardrop et un optimum social sur Pigou, comparer leurs coûts, interpréter le PoA et traiter les cas où le point critique sort du domaine faisable.

**Prérequis** : fonctions affines, dérivée d’une fonction quadratique et équilibre de Nash.

**Durée estimée** : 35 minutes.

Ce carnet ouvre une branche manquante de la série : les **jeux de routage / congestion à flux fractionnaire** et le **price of anarchy (PoA)** — l'écart entre l'équilibre où chaque usager minimise son propre coût (équilibre de Wardrop, qui est un équilibre de Nash du jeu de routage) et l'**optimum social** qui minimise le coût total.

C'est un approfondissement du palier 04 (Équilibre de Nash) vu du côté de la **nature** de l'équilibre : au lieu de demander *si* un équilibre existe, on mesure *à quel point* il est efficace. Un petit réseau concret, des coûts de lien fonctions de la charge, un équilibre d'utilisateur calculé, un optimum social calculé, et le **PoA mesuré** sur l'instance. On s'appuie sur nos fondations **forme normale** (GameTheory-02 / -04) : l'équilibre de Wardrop est un équilibre de Nash.

**Références** : Tim Roughgarden, [*Selfish Routing and the Price of Anarchy*](https://mitpress.mit.edu/9780262182430/selfish-routing-and-the-price-of-anarchy/) (MIT Press, 2005) ; Tim Roughgarden et Éva Tardos, [“How Bad is Selfish Routing?”](https://www.timroughgarden.org/papers/routing.pdf), *Journal of the ACM* 49(2), 236–259, 2002 (FOCS 2000). La borne 4/3 vaut sous les hypothèses précisées ci-dessous.

In [1]:
import math

def cout_congestionne(x, alpha=1.0):
    """Cout par usager d'un lien congestionne : c(x) = alpha * x."""
    return alpha * x

def cout_libre(x):
    """Cout par usager d'un lien 'libre' (cout constant, sans congestion)."""
    return 1.0

print('Briques du modele chargees :')
print('  lien congestionne : c(x) = alpha * x')
print('  lien libre        : c(x) = 1')

def verifier_domaine(alpha, D):
    if not (math.isfinite(alpha) and math.isfinite(D) and alpha > 0 and D > 0):
        raise ValueError("alpha et D doivent etre finis et strictement positifs")

def optimum_pigou(alpha, D):
    """Optimum social avec x_A contraint à [0,D] et coûts normalisés."""
    verifier_domaine(alpha, D)
    t = alpha * D
    x_A = min(D, 0.5 / alpha)
    x_B = D - x_A
    cout_total = D * (t if t <= 0.5 else 1.0 - 0.25 / t)
    return x_A, x_B, cout_total

def wardrop_pigou(alpha, D):
    """Equilibre Wardrop, flux conservés et coût social total."""
    verifier_domaine(alpha, D)
    t = alpha * D
    x_A = D if t <= 1.0 else min(D, 1.0 / alpha)
    x_B = D - x_A
    cout_total = D * (t if t <= 1.0 else 1.0)
    return x_A, x_B, cout_total

def poa_pigou(alpha, D=1.0):
    """PoA stable en unités normalisées, y compris les limites de flottants."""
    verifier_domaine(alpha, D)
    t = alpha * D
    return 1.0 if t <= 0.5 else min(t, 1.0) / (1.0 - 0.25 / t)


Briques du modele chargees :
  lien congestionne : c(x) = alpha * x
  lien libre        : c(x) = 1


## 1. L'instance canonique : l'exemple de Pigou

Deux chemins en parallèle relient la source `s` à la destination `t`, avec une demande totale `D` d'usagers :

- **Autoroute** (lien congestionné) : le coût par usager est proportionnel au nombre d'usagers dessus, `c_A(x_A) = alpha * x_A`.
- **Route locale** (lien libre) : coût constant `c_B = 1`, sans congestion.

Chaque usager choisit un chemin. C'est l'exemple fondateur qui donne son nom au phénomène : la route locale à coût constant égal à 1 conduit les usagers égoïstes à se jeter tous sur l'autoroute, alors que l'optimum social les répartirait.

In [2]:
alpha = 1.0   # pente de congestion de l'autoroute
D = 1.0       # demande totale (normalisee)

# OPTIMUM SOCIAL : min  alpha*x_A^2 + x_B   sous  x_A + x_B = D
#   (cout total = flux * cout par usager sur chaque lien)
#   condition du premier ordre : 2*alpha*x_A - 1 = 0  ->  x_A* = 1/(2*alpha)
xA_opt, xB_opt, cout_opt = optimum_pigou(alpha, D)

print(f"Optimum social : x_A = {xA_opt:.3f},  x_B = {xB_opt:.3f}")
print(f"Cout total a l'optimum  = {cout_opt:.4f}")

Optimum social : x_A = 0.500,  x_B = 0.500
Cout total a l'optimum  = 0.7500


## 2. L'équilibre de Wardrop (équilibre d'utilisateur / Nash)

À l'équilibre de Wardrop, **aucun usager ne peut réduire son coût personnel en changeant unilatéralement de chemin**. Tant que le coût de l'autoroute `c_A(x_A) = alpha * x_A` est inférieur au coût de la route locale (`1`), les usagers préfèrent l'autoroute : ils s'y massent jusqu'à ce que `alpha * x_A` atteigne `1`. Au-delà, plus personne n'y gagne à s'y ajouter.

Cet équilibre est un **équilibre de Nash** du jeu de routage : la stratégie de chaque joueur (son chemin) est une meilleure réponse à celle des autres.

In [3]:
# EQUILIBRE DE WARDROP : tous sur l'autoroute tant que c_A(D) = alpha*D <= 1
# (sinon l'autoroute coute plus que la route libre -> repartition jusqu'a c_A = 1)
xA_ue, xB_ue, cout_ue = wardrop_pigou(alpha, D)

print(f"Equilibre de Wardrop : x_A = {xA_ue:.3f},  x_B = {xB_ue:.3f}")
print(f"Cout total a l'equilibre = {cout_ue:.4f}")

Equilibre de Wardrop : x_A = 1.000,  x_B = 0.000
Cout total a l'equilibre = 1.0000


## 3. Le Price of Anarchy — mesuré

Le **price of anarchy** rapporte le coût à l'équilibre égoïste au coût à l'optimum social :

$$\text{PoA} = \frac{\text{cout total à l'équilibre de Wardrop}}{\text{cout total à l'optimum social}}$$

C'est le chiffre central du carnet : il **quantifie** le prix de l'intérêt individuel décentralisé. PoA = 1 signifie que l'équilibre est déjà optimal ; plus il est grand, plus l'égoïsme collective est coûteux.

In [4]:
poa = cout_ue / cout_opt

print(f"Price of Anarchy = {cout_ue:.4f} / {cout_opt:.4f} = {poa:.4f}")
print(f"  -> l'egarement interesse coute {100*(poa - 1):.1f} % de cout supplementaire")
print(f"  -> l'exemple de Pigou donne PoA = 4/3 = {4/3:.4f}  (coherence : {abs(poa - 4/3) < 1e-9})")

Price of Anarchy = 1.0000 / 0.7500 = 1.3333
  -> l'egarement interesse coute 33.3 % de cout supplementaire
  -> l'exemple de Pigou donne PoA = 4/3 = 1.3333  (coherence : True)


## 3bis. Le PoA en forme exacte — calculable à la main

Le chiffre central du carnet est **mesuré** en flottants à la section 3 : `cout_ue / cout_opt = 1.0000 / 0.7500 = 1.3333`, puis confronté à un littéral `4/3` codé en dur. Ce témoin **calcule la forme exacte** en arithmétique rationnelle (`fractions.Fraction`, sans flottant) à partir des données de l'instance canonique ($\alpha = 1$, $D = 1$), pour exhiber que le `1.3333` affiché **est précisément** la fraction $\frac{4}{3}$. Tout est **calculable à la main** : l'optimum social met $x_A^* = 1/(2\alpha) = 1/2$ (coût $3/4$), l'équilibre de Wardrop met tous les usagers sur l'autoroute ($\alpha D = 1 \le 1$, coût $1$), et le rapport des coûts totaux est $1 \div 3/4 = 4/3$.

> Les helpers `optimum_pigou` / `wardrop_pigou` du carnet renvoient des **flottants** ; la forme exacte n'est pas accessible par eux. Ce témoin recalcule la **même instance** en `Fraction` — ce n'est pas une réimplémentation du même calcul, c'est son changement de représentation (exact au lieu de flottant).

In [5]:
from fractions import Fraction

# Instance canonique (alpha = 1, D = 1) en arithmétique EXACTE.
alpha = Fraction(1)
D = Fraction(1)

# OPTIMUM SOCIAL :  x_A* = 1 / (2*alpha) ;  cout total = alpha*x_A*^2 + (D - x_A*)
xA_opt = 1 / (2 * alpha)
xB_opt = D - xA_opt
cout_opt = alpha * xA_opt**2 + xB_opt

# EQUILIBRE DE WARDROP : tous sur l'autoroute car alpha*D <= 1 ;  cout = alpha * x_A * D
xA_ue = D if alpha * D <= 1 else 1 / alpha
xB_ue = D - xA_ue
cout_ue = alpha * xA_ue * D

# PRICE OF ANARCHY — FORME EXACTE
poa_exact = cout_ue / cout_opt

print(f"Optimum social EXACT : x_A = {xA_opt},  x_B = {xB_opt},  cout total = {cout_opt}")
print(f"Equilibre Wardrop    : x_A = {xA_ue},  x_B = {xB_ue},  cout total = {cout_ue}")
print(f"Price of Anarchy EXACT = {cout_ue} / {cout_opt} = {poa_exact}")
print(f"  -> le 1.3333 flottant de la section 3 EST la fraction {poa_exact}")
print(f"  -> controle de coherence avec le flottant mesure : {abs(float(poa_exact) - cout_ue/cout_opt) < 1e-9}")

Optimum social EXACT : x_A = 1/2,  x_B = 1/2,  cout total = 3/4
Equilibre Wardrop    : x_A = 1,  x_B = 0,  cout total = 1
Price of Anarchy EXACT = 1 / 3/4 = 4/3
  -> le 1.3333 flottant de la section 3 EST la fraction 4/3
  -> controle de coherence avec le flottant mesure : True


### Lecture du résultat (forme exacte)

Le `1.3333` mesuré en flottant (section 3) s'identifie à la fraction exacte **$\frac{4}{3}$** : l'optimum social coûte $\frac{3}{4}$, l'équilibre égoïste coûte $1$, et leur rapport est $1 \div \frac{3}{4} = \frac{4}{3}$. C'est **calculable à la main** en deux lignes. L'exemple de Pigou atteint donc **exactement** la borne $\frac{4}{3}$ de Roughgarden–Tardos annoncée à la section 4 — c'est son **cas d'acuité** : le PoA du réseau n'est pas seulement *borné* à $\frac{4}{3}$, il l'**atteint**. Le chiffre flottant et la borne théorique sont le même nombre, et la fraction $\frac{4}{3}$ en est la forme exacte.

## 4. La borne 4/3 (Roughgarden–Tardos, 2002)

**Théorème.** Pour un réseau à **une seule commodity** dont les coûts de lien sont **affines non décroissantes** `c_e(x) = a_e x_e + b_e` avec `a_e >= 0` et `b_e >= 0` (modèle à flux fractionnaire), le price of anarchy est **borné par 4/3** :

$$\text{PoA} \le 4/3$$

Cette borne est **serrée** : l'instance de Pigou mesurée ci-dessus la **atteint exactement** en `α = 1` (PoA = 4/3) — elle en est le **cas d'égalité**. Sous les hypothèses précédentes, la borne générale vaut 4/3. (Le « 5/2 » souvent cité relève des jeux de congestion **atomiques** — joueurs indivisibles — de Christodoulou–Koutsoupias, un modèle distinct de celui de ce carnet.)

> **Portée du compagnon Lean** : il formalise un résultat atomique distinct : un équilibre minimisant le potentiel de Rosenthal a un coût au plus deux fois l’optimum, et ses fixtures montrent aussi un équilibre de Nash coûtant `15/7` fois l’optimum. Il ne formalise pas ici la borne non atomique 4/3.

## 5. Sensibilité : le PoA en fonction de la pente de congestion

Faisons varier la pente `alpha` de l'autoroute (à demande `D = 1` fixe) et **mesurons** le PoA. On attend : plus la congestion est raide, plus l'écart se referme — le PoA **décroît** vers 1. Le maximum tombe en `alpha = 1`, où Pigou réalise l'égalité de la borne ; sur tout ce domaine le PoA mesuré doit rester **≤ 4/3**.

In [6]:
import math

# Contrôles indépendants : ancrages usuels, frontières, valeurs extrêmes et domaine.
fixtures = [(1.0, 1.0, 4/3), (1.0, 0.1, 1.0), (0.25, 1.0, 1.0),
            (2.0, 2.0, 16/15), (1e-300, 1e-300, 1.0),
            (1e-300, 1e300, 4/3), (1e300, 1e300, 1.0)]
fixture_ok = all(math.isfinite(poa_pigou(a, d)) and abs(poa_pigou(a, d)-expected) < 1e-10
                 for a, d, expected in fixtures)
invalid_rejected = True
for bad_a, bad_d in [(0.0, 1.0), (1.0, float('inf')), (float('nan'), 1.0)]:
    try:
        poa_pigou(bad_a, bad_d)
        invalid_rejected = False
    except ValueError:
        pass
print(f"Contrôles bornés : {len(fixtures)} fixtures standard/extrêmes, PoA fini et oracle={fixture_ok}; paramètres invalides rejetés={invalid_rejected}.")

print(f"{'alpha':>6} | {'PoA mesure':>12} | {'<= 4/3 ?':>16}")
print("-" * 34)
for a in [1.0, 1.2, 1.4, 1.5, 1.8, 2.0]:
    p = poa_pigou(a)
    statut = "oui" if p <= 4.0/3.0 + 1e-9 else "non"
    print(f"{a:>6.1f} | {p:>12.4f} | {statut:>16}")


Contrôles bornés : 7 fixtures standard/extrêmes, PoA fini et oracle=True; paramètres invalides rejetés=True.
 alpha |   PoA mesure |         <= 4/3 ?
----------------------------------
   1.0 |       1.3333 |              oui
   1.2 |       1.2632 |              oui
   1.4 |       1.2174 |              oui
   1.5 |       1.2000 |              oui
   1.8 |       1.1613 |              oui
   2.0 |       1.1429 |              oui


La mesure confirme le profil théorique : le PoA est **maximal en `α = 1`** — valeur 4/3, le cas d'égalité de la borne — et **décroît** vers 1 quand la pente de congestion croît. La raison est directe : à `D = 1`, dès que `α > 1` l'équilibre se répartit et le coût d'équilibre est **figé à 1**, tandis que l'optimum social (`1 − 1/(4α)`) s'en approche. Tout le domaine reste donc **≤ 4/3**, jamais au-dessus : c'est exactement la borne de Roughgarden–Tardos, et le 4/3 n'est pas un artefact du point de bord — c'est la borne elle-même, atteinte en `α = 1`. Le PoA n'est pas un mot d'esprit : c'est une **quantité mesurable** qui structure la conception des réseaux.

## 5bis. La borne 4/3 n'est pas universelle : elle dépend du **degré** du coût

La borne 4/3 du §4 est énoncée sous l'hypothèse de coûts de lien **affines**
`c_e(x) = a_e x_e + b_e`. Tout ce que ce carnet a **exécuté** jusqu'ici est
l'instance affine de Pigou (`c_A(x) = alpha * x`) : cette hypothèse n'a donc
jamais été mise à l'épreuve par le code — elle est annoncée en prose, pas
vérifiée. Ce paragraphe rend cette **limite** vérifiable.

On généralise l'instance de Pigou à un lien de congestion **polynomial**
`c_A(x) = x^d` (degré `d`, demande `D = 1`), on re-dérive l'optimum social et
l'équilibre de Wardrop, puis on mesure le PoA.

**Proposition** : la borne 4/3 **cesse de tenir** dès que le degré `d` dépasse 1.
**Critère d'affaiblissement (posé avant le calcul)** : si le PoA du Pigou
quadratique (`d = 2`) **excède 4/3**, alors « PoA ≤ 4/3 » ne décrit que le
régime **affine** — la limite est l'hypothèse de linéarité elle-même. S'il
restait ≤ 4/3, la borne serait plus générale que l'hypothèse affirmée.

In [7]:
# Le meme reseau de Pigou, mais l'autoroute a un cout polynomial c(x) = x^d (degre d).
# On re-derive l'optimum social et l'equilibre de Wardrop, puis on mesure le PoA.
#   optimum social :  min  x_A^(d+1) + x_B   s.c.  x_A + x_B = D
#                     ->  FOC  (d+1) x_A^d - 1 = 0  ->  x_A* = (1/(d+1))^(1/d)
#   equilibre Wardrop :  tous sur l'autoroute tant que D^d <= 1
#                        (a D = 1 :  D^d = 1  ->  cout total = D * D^d = 1)
def poa_polynome(degre, D=1.0):
    """PoA de Pigou a cout polynomial c(x) = x^degre (demande D)."""
    if degre < 1:
        raise ValueError("degre doit etre >= 1")
    # optimum social (FOC : (d+1) x_A^d = 1, borne au flux faisable [0, D])
    xA_opt = min(D, (1.0 / (degre + 1.0)) ** (1.0 / degre))
    cout_opt = xA_opt ** (degre + 1) + (D - xA_opt)
    # equilibre de Wardrop (D^d <= 1 : tous sur l'autoroute ; sinon x_A = 1)
    if D ** degre <= 1.0:
        xA_ue, cout_ue = D, D * (D ** degre)
    else:
        xA_ue, cout_ue = 1.0, D * 1.0
    return cout_ue / cout_opt, xA_opt, cout_opt

# CONTROLE : le degre 1 (affine) doit reproduire EXACTEMENT le 4/3 du corps.
poa_d1, _, _ = poa_polynome(1)
# CONTRE-EXEMPLE : le degre 2 (quadratique) depasse 4/3.
poa_d2, xA_d2, opt_d2 = poa_polynome(2)

print(f"degre 1 (affine, controle) : PoA = {poa_d1:.4f}  [4/3 = {4/3:.4f}]  "
      f"coherent = {abs(poa_d1 - 4/3) < 1e-9}")
print(f"degre 2 (quadratique)      : PoA = {poa_d2:.4f}  [4/3 = {4/3:.4f}]  "
      f"depasse 4/3 = {poa_d2 > 4/3 + 1e-9}")
print(f"   x_A* = {xA_d2:.4f}  (= 1/sqrt(3) = {1/math.sqrt(3):.4f});  "
      f"cout optimum = {opt_d2:.4f}  (= 1 - 2*sqrt(3)/9 = {1 - 2*math.sqrt(3)/9:.4f})")
print()
print(f"{'degre':>5} | {'PoA mesure':>12} | {'<= 4/3 ?':>10}")
print("-" * 32)
for d in [1, 2, 3, 4, 5]:
    p, _, _ = poa_polynome(d)
    print(f"{d:>5} | {p:>12.4f} | {'oui' if p <= 4/3 + 1e-9 else 'NON':>10}")

degre 1 (affine, controle) : PoA = 1.3333  [4/3 = 1.3333]  coherent = True
degre 2 (quadratique)      : PoA = 1.6258  [4/3 = 1.3333]  depasse 4/3 = True
   x_A* = 0.5774  (= 1/sqrt(3) = 0.5774);  cout optimum = 0.6151  (= 1 - 2*sqrt(3)/9 = 0.6151)

degre |   PoA mesure |   <= 4/3 ?
--------------------------------
    1 |       1.3333 |        oui
    2 |       1.6258 |        NON
    3 |       1.8956 |        NON
    4 |       2.1505 |        NON
    5 |       2.3944 |        NON


### Lecture du résultat

Le contrôle `degré 1` reproduit **exactement** le 4/3 du corps (33,3 %
d'égarement, comme au §3) — la généralisation est donc fidèle. Le
**contre-exemple** est net : dès le `degré 2` (coût quadratique), le PoA saute
à **1,63**, au-dessus de 4/3, et **croît** avec le degré (1,90 à `d = 3`, 2,15
à `d = 4`, 2,39 à `d = 5`).

La borne 4/3 du §4 n'est donc **pas** une propriété universelle des jeux de
routage : c'est une propriété de la **classe affine** (degré 1). La constante
4/3 est **bornée au degré** du coût de lien — l'hypothèse « coûts affines » du
§4 est **porteuse**, pas décorative. C'est précisément ce que rendait invisible
le fait que le carnet n'exécutait que l'instance affine : la **limite** de la
borne (ce qui la fait cesser de tenir) n'apparaît qu'en **sortant** de la
classe, et ici elle est **mesurée**, pas affirmée.

## 6. Exercices

### Exercice 1 — Équilibre de Wardrop général

Pour une instance à **2 liens en parallèle** (autoroute `c(x) = alpha*x`, route libre `c = 1`) et une demande `D`, calcule l'équilibre de Wardrop, c'est-à-dire la répartition `(x_A, x_B)` des usagers à l'équilibre.

**Indice** : à l'équilibre, un usager ne gagne rien à changer de lien — compare le coût `c_A(x_A)` de l'autoroute au coût `1` de la route locale.

In [8]:
def wardrop_2liens(alpha, D):
    """Renvoie (x_A, x_B) a l'equilibre de Wardrop de l'instance a 2 liens.

    Autoroute : c(x) = alpha*x.  Route libre : c = 1.  Demande totale : D.
    """
    # TODO etudiant
    print("Exercice à compléter")
    return None  # TODO: (x_A, x_B)
print("Exercice 1 : fonction Wardrop stub conservee.")


Exercice 1 : fonction Wardrop stub conservee.


### Exercice 2 — La fonction PoA

Écris une fonction `poa(alpha, D)` qui renvoie le **price of anarchy mesuré** de l'instance de Pigou généralisée (autoroute `alpha*x`, route libre `1`, demande `D`). Elle doit reproduire la valeur `4/3` pour `alpha = 1, D = 1`.

In [9]:
def poa(alpha, D):
    """Renvoie le price of anarchy (cout Wardrop / cout optimum social).

    Instance : autoroute c(x) = alpha*x, route libre c = 1, demande D.
    Indice : recompose l'optimum social (FOC) et l'equilibre de Wardrop.
    """
    # TODO etudiant
    print("Exercice à compléter")
    return None  # TODO: float
print("Exercice 2 : fonction PoA stub conservee.")


Exercice 2 : fonction PoA stub conservee.


### Exercice 3 — Le PoA en fonction de la demande

Pour l'instance canonique (`alpha = 1`), calcule le PoA pour une **série de valeurs de la demande** `D` sur `[1, 4]` et décris la tendance. Que devient le PoA quand la demande croît ? (Rappel : à l'équilibre, tous sont sur l'autoroute tant que `alpha * D <= 1` ; au-delà, le flux se répartit.)

In [10]:
def poa_en_fonction_de_D(D, alpha=1.0):
    """Renvoie le PoA de l'instance canonique (alpha=1) pour une demande D."""
    # TODO etudiant
    print("Exercice à compléter")
    return None  # TODO: float

# (a completer) evaluer sur D = [1.0, 2.0, 3.0, 4.0] et afficher la tendance
print("Exercice 3 : calcul de sensibilite a completer.")


Exercice 3 : calcul de sensibilite a completer.


## Conclusion

### À retenir

Le **price of anarchy** transforme une intuition (« les usagers égoïstes gâchent le réseau ») en **quantité mesurable** : sur l'exemple de Pigou, l'intérêt individuel décentralisé coûte **33 %** de supplément (PoA = 4/3) par rapport à l'optimum social, et la borne 4/3 de Roughgarden–Tardos, sous les hypothèses `a_e,b_e >= 0`, encadre ce prix — Pigou en est le cas d’égalité. Le compagnon Lean formalise un résultat atomique distinct : borne 2× pour un équilibre minimisant le potentiel de Rosenthal, avec témoin d’un équilibre coûtant 15/7 fois l’optimum ; il ne formalise pas la borne non atomique 4/3. C'est le pont entre la **théorie des jeux** (l'équilibre de Nash / Wardrop) et la **conception d'algorithmes et de réseaux** efficaces — la branche appliquée que ce carnet ajoute à la série.

## 8. Récapitulatif des exercices

1. [Équilibre de Wardrop général](#exercice-1--équilibre-de-wardrop-général) : flux conservés et conditions de coût sur les chemins utilisés.
2. [Fonction PoA](#exercice-2--la-fonction-poa) : quotient stable des coûts pour des paramètres intérieurs, de frontière et extrêmes.
3. [Variation de la demande](#exercice-3--le-poa-en-fonction-de-la-demande) : comparer les valeurs calculées pour `D = 1, 2, 3, 4`.

Les trois fonctions restent à compléter. Les valeurs de contrôle ne remplissent aucun stub.

## Vérifiez votre compréhension

1. Pourquoi faut-il borner l’optimum par D ?
2. Quel est le ratio canonique et quelle est la portée du compagnon Lean ?

<details>
<summary>Éléments de réponse</summary>

1. Un flux sur un chemin ne peut dépasser la demande totale ; si le point critique est au-delà, le minimum faisable se trouve en `x_A=D`.
2. Le ratio non atomique de Pigou vaut `1/0,75=4/3`. Le compagnon Lean formalise un jeu atomique et sa borne de potentiel 2×, et présente un témoin à 15/7 pour un autre équilibre de Nash.

</details>

## Pour aller plus loin

- [Jumeau C#](GameTheory-04h-Routing-PriceOfAnarchy-CSharp.ipynb)
- [Compagnon Lean](GameTheory-04h-Routing-PriceOfAnarchy-Lean.ipynb)
- Roughgarden et Tardos (2002), *How Bad is Selfish Routing?*.
